# 03 — Supervisor-facing report

This notebook is presentation-only. It reads dataset, PySR-run, and AUC-analysis artifacts. It does not fit models or recalculate metrics. All scientific content remains provisional, unverified, and pending review.

In [ ]:
from datetime import datetime, timezone
import json
from pathlib import Path

import matplotlib.pyplot as plt
import pandas as pd
from IPython.display import Markdown, display

def repository_root() -> Path:
    candidate = Path.cwd().resolve()
    for _ in range(6):
        if (candidate / 'data' / 'raw').is_dir():
            return candidate
        candidate = candidate.parent
    raise FileNotFoundError('Could not locate repository root from notebook directory.')

ROOT = repository_root()
RUN_ID = 'notebook_pysr_run_v1'
RUN_DIR = ROOT / 'outputs' / 'pysr' / RUN_ID
AUC_DIR = ROOT / 'outputs' / 'auc' / RUN_ID
REPORT_DIR = ROOT / 'outputs' / 'report'
print(f'Repository: {ROOT}')
print(f'Run directory: {RUN_DIR}')
print(f'AUC directory: {AUC_DIR}')

In [ ]:
def read_json(path: Path):
    return json.loads(path.read_text(encoding='utf-8')) if path.is_file() else None

run_metadata = read_json(RUN_DIR / 'run_metadata.json')
auc_metrics = read_json(AUC_DIR / 'metrics.json')
dataset_readme = (ROOT / 'data' / 'README.md').read_text(encoding='utf-8')
missing = [
    str(path.relative_to(ROOT))
    for path in [RUN_DIR / 'run_metadata.json', AUC_DIR / 'metrics.json', AUC_DIR / 'roc_curve.csv']
    if not path.is_file()
]
if missing:
    display(Markdown('**Missing evidence artifacts:**\n\n' + '\n'.join(f'- `{item}`' for item in missing)))
else:
    display(Markdown('All required report inputs are present.'))

if run_metadata:
    display(pd.DataFrame([
        {'field': key, 'value': value}
        for key, value in run_metadata.items()
        if key in {'run_id', 'dataset_id', 'raw_path', 'features', 'target', 'positive_label', 'split', 'seeds', 'operators', 'search', 'fit_status', 'review_status'}
    ]))
if auc_metrics:
    display(pd.DataFrame([auc_metrics]))
display(Markdown('**Review status:** provisional, unverified, pending review.'))

In [ ]:
roc_path = AUC_DIR / 'roc_curve.csv'
if roc_path.is_file():
    roc = pd.read_csv(roc_path)
    fig, ax = plt.subplots(figsize=(6, 4))
    ax.plot(roc['fpr'], roc['tpr'], color='#0f766e', linewidth=2)
    ax.plot([0, 1], [0, 1], '--', color='0.5')
    ax.set(xlabel='False positive rate', ylabel='True positive rate', title=f'Saved continuous-score ROC — {RUN_ID}')
    ax.grid(alpha=0.25)
    fig.tight_layout()
    plt.show()
else:
    display(Markdown('ROC figure unavailable because the AUC analysis output is missing.'))

REPORT_DIR.mkdir(parents=True, exist_ok=True)
manifest = {
    'run_id': RUN_ID,
    'run_metadata': str((RUN_DIR / 'run_metadata.json').relative_to(ROOT)),
    'auc_metrics': str((AUC_DIR / 'metrics.json').relative_to(ROOT)),
    'report_status': 'provisional, unverified, pending review',
    'created_utc': datetime.now(timezone.utc).isoformat(),
    'missing_inputs': missing,
}
(REPORT_DIR / 'report_manifest.json').write_text(json.dumps(manifest, indent=2) + '\n', encoding='utf-8')
print(f"Report manifest written to {REPORT_DIR / 'report_manifest.json'}")